# 05 — Model Training

Logistic Regression (interpretable baseline) vs. Random Forest vs. XGBoost, all on the exact same feature matrix and the exact same group-aware train/test split (`src/models.py`).

**Why group-aware splitting matters here:** 23% of patients have more than one encounter (`01_data_audit.ipynb`). A plain random row split would let the same patient's encounters land in both train and test, which leaks patient-specific signal across the split and inflates reported performance. `GroupShuffleSplit` on `patient_nbr` guarantees no patient crosses the boundary.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import joblib
from models import train_all, get_feature_matrix, NUMERIC_FEATURES, CATEGORICAL_FEATURES

df = pd.read_csv('../data/processed/diabetic_data_features.csv')
print(f'{len(NUMERIC_FEATURES)} numeric features, {len(CATEGORICAL_FEATURES)} categorical features')

19 numeric features, 35 categorical features


In [2]:
fitted, (X_train, X_test, y_train, y_test) = train_all(df)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train positive rate: {y_train.mean():.3f}, Test positive rate: {y_test.mean():.3f}')
print('Models trained:', list(fitted.keys()))

Train: (80039, 54), Test: (20075, 54)
Train positive rate: 0.113, Test positive rate: 0.115
Models trained: ['logistic_regression', 'random_forest', 'xgboost']


Class imbalance (≈11% positive) is handled two ways: `class_weight="balanced"` for Logistic Regression / Random Forest, and `scale_pos_weight` (ratio of negative:positive in the training fold) for XGBoost — not by oversampling/undersampling, which would distort the base rate used later in the optimization layer's expected-value calculations.

In [3]:
import os
os.makedirs('../data/processed/models', exist_ok=True)
for name, pipe in fitted.items():
    joblib.dump(pipe, f'../data/processed/models/{name}.joblib')
joblib.dump((X_train, X_test, y_train, y_test), '../data/processed/models/splits.joblib')
print('Saved fitted pipelines and train/test split to data/processed/models/')

Saved fitted pipelines and train/test split to data/processed/models/
